In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from urllib.parse import urlparse, urlunparse, parse_qsl, urlencode
import re

# ===== RUTA BASE =====
base_dir = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Limpieza de datos y scrapping")
base_dir.mkdir(parents=True, exist_ok=True)

# ===== ARCHIVOS DE ENTRADA =====
pc_file = base_dir / "pc componentes scrapping limpio.csv"
kg_file = base_dir / "dataset amazon electronics 2025 limpio.csv"

# ===== ARCHIVO DE SALIDA =====
macro_file = base_dir / "Dataset Maestro.csv"

# ===== CARGA =====
print("📥 Cargando datasets...")
pc = pd.read_csv(pc_file)
kg = pd.read_csv(kg_file)

print(f"  - PC Componentes: {len(pc)} productos")
print(f"  - Kaggle Amazon: {len(kg)} productos")

# Normalizar nombres de columnas
pc.columns = pc.columns.str.strip().str.lower()
kg.columns = kg.columns.str.strip().str.lower()

# ===== LIMPIEZA BÁSICA DE TEXTO =====
def clean_text_series(s):
    return s.astype("string").str.strip()

# ===== LIMPIEZA / NORMALIZACIÓN DE URLS =====
def normalize_url(url):
    if pd.isna(url) or str(url).strip() == "":
        return pd.NA
    
    url = str(url).strip()
    
    md_match = re.search(r"\((https?://[^)]+)\)", url)
    if md_match:
        url = md_match.group(1)
    
    url = url.strip("[]")
    
    if url.startswith("www."):
        url = "https://" + url
    elif not url.startswith(("http://", "https://")):
        url = "https://" + url.lstrip("/")
    
    parsed = urlparse(url)
    
    scheme = "https"
    netloc = parsed.netloc.lower().replace("www.", "")
    path = parsed.path.replace("//", "/").strip()
    
    while "//" in path:
        path = path.replace("//", "/")
    
    if path != "/":
        path = path.rstrip("/")
    
    query_params = parse_qsl(parsed.query, keep_blank_values=True)
    query_params = [(k, v) for k, v in query_params if not k.lower().startswith("utm_")]
    query = urlencode(sorted(query_params))
    
    return urlunparse((scheme, netloc, path, "", query, ""))

# ===== ESTANDARIZAR PC COMPONENTES =====
print("\n🔧 Estandarizando PC Componentes...")
pc_macro = pd.DataFrame({
    "source": "pccomponentes",
    "product_name": pc["nombre"] if "nombre" in pc.columns else np.nan,
    "brand": pc["marca"] if "marca" in pc.columns else np.nan,
    "category": pc["subcategoria"] if "subcategoria" in pc.columns else np.nan,
    "subcategory": pc["subcategoria"] if "subcategoria" in pc.columns else np.nan,
    "price": pc["precio_actual"] if "precio_actual" in pc.columns else np.nan,
    "price_original": pc["precio_original"] if "precio_original" in pc.columns else np.nan,
    "discount_pct": pc["descuento_pct"] if "descuento_pct" in pc.columns else np.nan,
    "rating": pc["rating"] if "rating" in pc.columns else np.nan,
    "review_count": pc["num_opiniones"] if "num_opiniones" in pc.columns else np.nan,
    "popularity": pc["num_opiniones"] if "num_opiniones" in pc.columns else np.nan,
    "product_url": pc["product_url"] if "product_url" in pc.columns else np.nan,
    "seller": pc["seller_raw"] if "seller_raw" in pc.columns else np.nan,
    "promo_tag": pc["promo_raw"] if "promo_raw" in pc.columns else np.nan,
    "sku": pc["sku"] if "sku" in pc.columns else np.nan,
})

# ===== ESTANDARIZAR KAGGLE AMAZON =====
print("🔧 Estandarizando Kaggle Amazon...")
kg_macro = pd.DataFrame({
    "source": "kaggle_amazon",
    "product_name": kg["product_name"] if "product_name" in kg.columns else kg["name"] if "name" in kg.columns else np.nan,
    "brand": kg["brand"] if "brand" in kg.columns else np.nan,
    "category": kg["category"] if "category" in kg.columns else np.nan,
    "subcategory": kg["subcategory"] if "subcategory" in kg.columns else np.nan,
    "price": kg["price"] if "price" in kg.columns else np.nan,
    "price_original": kg["price_original"] if "price_original" in kg.columns else np.nan,
    "discount_pct": kg["discount_pct"] if "discount_pct" in kg.columns else np.nan,
    "rating": kg["rating"] if "rating" in kg.columns else np.nan,
    "review_count": kg["review_count"] if "review_count" in kg.columns else np.nan,
    "popularity": kg["sales"] if "sales" in kg.columns else kg["popularity"] if "popularity" in kg.columns else np.nan,
    "product_url": kg["product_url"] if "product_url" in kg.columns else np.nan,
    "seller": kg["seller"] if "seller" in kg.columns else np.nan,
    "promo_tag": kg["promo_tag"] if "promo_tag" in kg.columns else np.nan,
    "sku": kg["sku"] if "sku" in kg.columns else np.nan,
})

# ===== CONCATENAR =====
print("\n🔗 Concatenando datasets...")
macro = pd.concat([pc_macro, kg_macro], ignore_index=True)

# ===== NORMALIZAR TIPO DE DATOS =====
for col in ["price", "price_original", "discount_pct", "rating", "review_count", "popularity"]:
    macro[col] = pd.to_numeric(macro[col], errors="coerce")

for col in ["product_name", "brand", "category", "subcategory", "product_url", "seller", "promo_tag", "source", "sku"]:
    macro[col] = clean_text_series(macro[col])

# ===== LIMPIEZA DE URLS =====
print("🔗 Normalizando URLs...")
macro["product_url_clean"] = macro["product_url"].apply(normalize_url)
macro["url_mal_formateada"] = macro["product_url_clean"].isna()
macro["product_url"] = macro["product_url_clean"]
macro = macro.drop(columns=["product_url_clean"])

# ===== LIMPIEZA DE NULOS BÁSICOS =====
antes = len(macro)
macro = macro.dropna(subset=["product_name"], how="all").copy()
macro = macro.dropna(subset=["price"], how="all").copy()
print(f"🗑️ Eliminadas {antes - len(macro)} filas sin nombre o precio")

# ===== NORMALIZAR CAMPOS PARA DUPLICADOS =====
macro["product_name_norm"] = (
    macro["product_name"]
    .astype("string")
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

macro["brand_norm"] = (
    macro["brand"]
    .astype("string")
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

macro["subcategory_norm"] = (
    macro["subcategory"]
    .astype("string")
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# ===== DUPLICADOS EVIDENTES =====
macro["duplicado_evidente"] = macro.duplicated(
    subset=["source", "product_name_norm", "brand_norm", "price"],
    keep=False
)

macro["duplicado_url"] = macro.duplicated(
    subset=["source", "product_url"],
    keep=False
)

antes = len(macro)
macro = macro.drop_duplicates(
    subset=["source", "product_name_norm", "brand_norm", "price"],
    keep="first"
).copy()
print(f"🗑️ Eliminados {antes - len(macro)} duplicados")

# ===== LIMPIEZA FINAL DE COLUMNAS AUXILIARES =====
macro = macro.drop(
    columns=[
        "product_name_norm",
        "brand_norm",
        "subcategory_norm",
        "duplicado_evidente"
    ],
    errors="ignore"
)

# ===== ORDENAR COLUMNAS =====
ordered_cols = [
    "source", "product_name", "brand", "category", "subcategory",
    "price", "price_original", "discount_pct", "rating",
    "review_count", "popularity", "seller", "promo_tag",
    "product_url", "sku", "url_mal_formateada", "duplicado_url"
]
macro = macro[[c for c in ordered_cols if c in macro.columns]].copy()

# ===== GUARDAR =====
macro.to_csv(macro_file, index=False, encoding="utf-8-sig")

print(f"\n{'='*60}")
print(f"✅ DATASET MAESTRO CREADO")
print(f"{'='*60}")
print(f"📁 Guardado en: {macro_file}")
print(f"📊 Filas totales: {len(macro)}")
print(f"\n=== DISTRIBUCIÓN POR FUENTE ===")
print(macro['source'].value_counts())
print(f"\n=== RESUMEN DE CALIDAD ===")
print(f"Productos con rating: {macro['rating'].notna().sum()} ({macro['rating'].notna().sum()/len(macro)*100:.1f}%)")
print(f"Productos con opiniones: {macro['review_count'].notna().sum()} ({macro['review_count'].notna().sum()/len(macro)*100:.1f}%)")
print(f"\n=== Primeras 5 filas ===")
print(macro[['source', 'product_name', 'brand', 'price', 'rating', 'review_count']].head())

📥 Cargando datasets...
  - PC Componentes: 231 productos
  - Kaggle Amazon: 29 productos

🔧 Estandarizando PC Componentes...
🔧 Estandarizando Kaggle Amazon...

🔗 Concatenando datasets...
🔗 Normalizando URLs...
🗑️ Eliminadas 29 filas sin nombre o precio
🗑️ Eliminados 0 duplicados

✅ DATASET MAESTRO CREADO
📁 Guardado en: C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Limpieza de datos y scrapping\Dataset Maestro.csv
📊 Filas totales: 231

=== DISTRIBUCIÓN POR FUENTE ===
source
pccomponentes    231
Name: count, dtype: Int64

=== RESUMEN DE CALIDAD ===
Productos con rating: 0 (0.0%)
Productos con opiniones: 231 (100.0%)

=== Primeras 5 filas ===
          source                                       product_name  brand  \
0  pccomponentes  PcCom Ready AMD Ryzen 7 5800X / 32GB / 1TB SSD...  PcCom   
1  pccomponentes  PcCom Imperial AMD Ryzen 7 5800X / 32GB / 2TB ...  PcCom   
2  pccomponentes  PcCom Imperial AMD Ryzen 7 5800X / 32GB / 2TB ...  PcCom   
3  pccomp

In [2]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# ====== RUTA DE SALIDA ======
output_dir = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Limpieza de datos y scrapping")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "pc componentes scrapping limpio.csv"

# ====== CARGA DEL SCRAPING BRUTO ======
input_file = output_dir / "pccomponentes_gaming_scraping_raw.csv"

print(f"📥 Buscando archivo: {input_file}")
print(f"¿Existe? {input_file.exists()}")

if not input_file.exists():
    print("\n❌ ERROR: No se encontró el archivo del scraping nuevo")
    print("Debes ejecutar primero el script de scraping que te funcionó")
    exit()

df = pd.read_csv(input_file)

print(f"📥 Cargados {len(df)} productos del scraping raw")
print(f"Productos con rating: {df['rating'].notna().sum()}")
print(f"Productos con opiniones: {df['num_opiniones'].notna().sum()}")

# ====== FUNCIONES DE LIMPIEZA ======
def limpiar_url(url):
    if pd.isna(url) or url is None:
        return np.nan
    url = str(url).strip()
    m = re.search(r"https?://[^\)\]\s]+", url)
    return m.group(0) if m else url

def limpiar_texto(x):
    if pd.isna(x) or x is None:
        return np.nan
    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)
    return x if x else np.nan

def normalizar_numero(x):
    if pd.isna(x) or x is None:
        return np.nan
    return float(x) if not pd.isna(x) else np.nan

def normalizar_entero(x):
    if pd.isna(x) or x is None:
        return np.nan
    return int(x) if not pd.isna(x) else np.nan

# ====== LIMPIEZA GENERAL ======
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].apply(limpiar_texto)

# URLs
if "product_url" in df.columns:
    df["product_url"] = df["product_url"].apply(limpiar_url)

if "url_categoria" in df.columns:
    df["url_categoria"] = df["url_categoria"].apply(limpiar_url)

# Normalizar numéricos
for col in ["precio_actual", "precio_original", "descuento_pct", "rating", "num_opiniones"]:
    if col in df.columns:
        df[col] = df[col].apply(normalizar_numero)

if "sku" in df.columns:
    df["sku"] = df["sku"].apply(normalizar_entero)

# ====== QUITAR DUPLICADOS ======
subset_cols = [c for c in ["sku", "product_url", "nombre"] if c in df.columns]
if subset_cols:
    antes = len(df)
    df = df.drop_duplicates(subset=subset_cols, keep="first").copy()
    print(f"🗑️ Eliminados {antes - len(df)} duplicados")

# ====== QUITAR FILAS VACÍAS/INÚTILES ======
cond_basica = pd.Series(True, index=df.index)

if "nombre" in df.columns:
    cond_basica &= df["nombre"].notna()

if "product_url" in df.columns:
    cond_basica &= df["product_url"].notna()

df = df[cond_basica].copy()
df = df.dropna(how="all").copy()

# ====== LIMPIEZA DE VALORES MALOS ======
if "precio_actual" in df.columns:
    df.loc[df["precio_actual"] <= 0, "precio_actual"] = np.nan

if "precio_original" in df.columns:
    df.loc[df["precio_original"] <= 0, "precio_original"] = np.nan

if "descuento_pct" in df.columns:
    df.loc[(df["descuento_pct"] < 0) | (df["descuento_pct"] > 90), "descuento_pct"] = np.nan

if "rating" in df.columns:
    df.loc[(df["rating"] < 0) | (df["rating"] > 5), "rating"] = np.nan

if "num_opiniones" in df.columns:
    df.loc[df["num_opiniones"] < 0, "num_opiniones"] = np.nan

# ====== ORDEN DE COLUMNAS ======
cols_priority = [
    "sku", "nombre", "marca", "subcategoria", "categoria_raw",
    "precio_actual", "precio_original", "descuento_pct",
    "rating", "num_opiniones",
    "seller_raw", "promo_raw",
    "product_url", "url_categoria"
]

cols_existing = [c for c in cols_priority if c in df.columns]
other_cols = [c for c in df.columns if c not in cols_existing]
df = df[cols_existing + other_cols].copy()

# ====== GUARDAR CSV LIMPIO ======
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"\n{'='*60}")
print(f"✅ ARCHIVO LIMPIO GUARDADO")
print(f"{'='*60}")
print(f"📁 Guardado en: {output_file}")
print(f"📊 Filas finales: {len(df)}")
print(f"\n=== RESUMEN DE DATOS ===")
print(f"Productos con rating: {df['rating'].notna().sum()} ({df['rating'].notna().sum()/len(df)*100:.1f}%)")
print(f"Productos con opiniones: {df['num_opiniones'].notna().sum()} ({df['num_opiniones'].notna().sum()/len(df)*100:.1f}%)")
print(f"Rating promedio: {df['rating'].mean():.2f}")
print(f"Opiniones promedio: {df['num_opiniones'].mean():.0f}")
print(f"\n=== Primeras 5 filas ===")
print(df[['nombre', 'marca', 'subcategoria', 'precio_actual', 'rating', 'num_opiniones']].head())

📥 Buscando archivo: C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Limpieza de datos y scrapping\pccomponentes_gaming_scraping_raw.csv
¿Existe? True
📥 Cargados 245 productos del scraping raw
Productos con rating: 245
Productos con opiniones: 245
🗑️ Eliminados 14 duplicados

✅ ARCHIVO LIMPIO GUARDADO
📁 Guardado en: C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Limpieza de datos y scrapping\pc componentes scrapping limpio.csv
📊 Filas finales: 231

=== RESUMEN DE DATOS ===
Productos con rating: 231 (100.0%)
Productos con opiniones: 231 (100.0%)
Rating promedio: 4.55
Opiniones promedio: 1262

=== Primeras 5 filas ===
                                              nombre  marca subcategoria  \
0  PcCom Ready AMD Ryzen 7 5800X / 32GB / 1TB SSD...  PcCom           PC   
1  PcCom Imperial AMD Ryzen 7 5800X / 32GB / 2TB ...  PcCom           PC   
2  PcCom Imperial AMD Ryzen 7 5800X / 32GB / 2TB ...  PcCom           PC   
3  PcCom Imper

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from urllib.parse import urlparse, urlunparse, parse_qsl, urlencode
import re

# ===== RUTA BASE =====
base_dir = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Limpieza de datos y scrapping")
base_dir.mkdir(parents=True, exist_ok=True)

# ===== ARCHIVOS DE ENTRADA =====
pc_file = base_dir / "pc componentes scrapping limpio.csv"
kg_file = base_dir / "dataset amazon electronics 2025 limpio.csv"

# ===== ARCHIVO DE SALIDA =====
macro_file = base_dir / "Dataset Definitivo.csv"

# ===== CARGA =====
print("📥 Cargando datasets...")
pc = pd.read_csv(pc_file)
kg = pd.read_csv(kg_file)

print(f"  - PC Componentes: {len(pc)} productos")
print(f"  - Kaggle Amazon: {len(kg)} productos")

# Normalizar nombres de columnas
pc.columns = pc.columns.str.strip().str.lower()
kg.columns = kg.columns.str.strip().str.lower()

# ===== LIMPIEZA BÁSICA DE TEXTO =====
def clean_text_series(s):
    return s.astype("string").str.strip()

# ===== LIMPIEZA / NORMALIZACIÓN DE URLS =====
def normalize_url(url):
    if pd.isna(url) or str(url).strip() == "":
        return pd.NA
    
    url = str(url).strip()
    
    md_match = re.search(r"\((https?://[^)]+)\)", url)
    if md_match:
        url = md_match.group(1)
    
    url = url.strip("[]")
    
    if url.startswith("www."):
        url = "https://" + url
    elif not url.startswith(("http://", "https://")):
        url = "https://" + url.lstrip("/")
    
    parsed = urlparse(url)
    
    scheme = "https"
    netloc = parsed.netloc.lower().replace("www.", "")
    path = parsed.path.replace("//", "/").strip()
    
    while "//" in path:
        path = path.replace("//", "/")
    
    if path != "/":
        path = path.rstrip("/")
    
    query_params = parse_qsl(parsed.query, keep_blank_values=True)
    query_params = [(k, v) for k, v in query_params if not k.lower().startswith("utm_")]
    query = urlencode(sorted(query_params))
    
    return urlunparse((scheme, netloc, path, "", query, ""))

# ===== ESTANDARIZAR PC COMPONENTES =====
print("\n🔧 Estandarizando PC Componentes...")
pc_macro = pd.DataFrame({
    "source": "pccomponentes",
    "product_name": pc["nombre"] if "nombre" in pc.columns else np.nan,
    "brand": pc["marca"] if "marca" in pc.columns else np.nan,
    "category": pc["subcategoria"] if "subcategoria" in pc.columns else np.nan,
    "subcategory": pc["subcategoria"] if "subcategoria" in pc.columns else np.nan,
    "price": pc["precio_actual"] if "precio_actual" in pc.columns else np.nan,
    "price_original": pc["precio_original"] if "precio_original" in pc.columns else np.nan,
    "discount_pct": pc["descuento_pct"] if "descuento_pct" in pc.columns else np.nan,
    "rating": pc["rating"] if "rating" in pc.columns else np.nan,
    "review_count": pc["num_opiniones"] if "num_opiniones" in pc.columns else np.nan,
    "popularity": pc["num_opiniones"] if "num_opiniones" in pc.columns else np.nan,
    "product_url": pc["product_url"] if "product_url" in pc.columns else np.nan,
    "seller": pc["seller_raw"] if "seller_raw" in pc.columns else np.nan,
    "promo_tag": pc["promo_raw"] if "promo_raw" in pc.columns else np.nan,
    "sku": pc["sku"] if "sku" in pc.columns else np.nan,
})

# ===== ESTANDARIZAR KAGGLE AMAZON =====
print("🔧 Estandarizando Kaggle Amazon...")
kg_macro = pd.DataFrame({
    "source": "kaggle_amazon",
    "product_name": kg["product_name"] if "product_name" in kg.columns else kg["name"] if "name" in kg.columns else np.nan,
    "brand": kg["brand"] if "brand" in kg.columns else np.nan,
    "category": kg["category"] if "category" in kg.columns else np.nan,
    "subcategory": kg["subcategory"] if "subcategory" in kg.columns else np.nan,
    "price": kg["price"] if "price" in kg.columns else np.nan,
    "price_original": kg["price_original"] if "price_original" in kg.columns else np.nan,
    "discount_pct": kg["discount_pct"] if "discount_pct" in kg.columns else np.nan,
    "rating": kg["rating"] if "rating" in kg.columns else np.nan,
    "review_count": kg["review_count"] if "review_count" in kg.columns else np.nan,
    "popularity": kg["sales"] if "sales" in kg.columns else kg["popularity"] if "popularity" in kg.columns else np.nan,
    "product_url": kg["product_url"] if "product_url" in kg.columns else np.nan,
    "seller": kg["seller"] if "seller" in kg.columns else np.nan,
    "promo_tag": kg["promo_tag"] if "promo_tag" in kg.columns else np.nan,
    "sku": kg["sku"] if "sku" in kg.columns else np.nan,
})

# ===== CONCATENAR =====
print("\n🔗 Concatenando datasets...")
macro = pd.concat([pc_macro, kg_macro], ignore_index=True)

# ===== NORMALIZAR TIPO DE DATOS =====
for col in ["price", "price_original", "discount_pct", "rating", "review_count", "popularity"]:
    macro[col] = pd.to_numeric(macro[col], errors="coerce")

for col in ["product_name", "brand", "category", "subcategory", "product_url", "seller", "promo_tag", "source", "sku"]:
    macro[col] = clean_text_series(macro[col])

# ===== LIMPIEZA DE URLS =====
print("🔗 Normalizando URLs...")
macro["product_url_clean"] = macro["product_url"].apply(normalize_url)
macro["url_mal_formateada"] = macro["product_url_clean"].isna()
macro["product_url"] = macro["product_url_clean"]
macro = macro.drop(columns=["product_url_clean"])

# ===== LIMPIEZA DE NULOS BÁSICOS =====
antes = len(macro)
macro = macro.dropna(subset=["product_name"], how="all").copy()
macro = macro.dropna(subset=["price"], how="all").copy()
print(f"🗑️ Eliminadas {antes - len(macro)} filas sin nombre o precio")

# ===== NORMALIZAR CAMPOS PARA DUPLICADOS =====
macro["product_name_norm"] = (
    macro["product_name"]
    .astype("string")
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

macro["brand_norm"] = (
    macro["brand"]
    .astype("string")
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

macro["subcategory_norm"] = (
    macro["subcategory"]
    .astype("string")
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# ===== DUPLICADOS EVIDENTES =====
macro["duplicado_evidente"] = macro.duplicated(
    subset=["source", "product_name_norm", "brand_norm", "price"],
    keep=False
)

macro["duplicado_url"] = macro.duplicated(
    subset=["source", "product_url"],
    keep=False
)

antes = len(macro)
macro = macro.drop_duplicates(
    subset=["source", "product_name_norm", "brand_norm", "price"],
    keep="first"
).copy()
print(f"🗑️ Eliminados {antes - len(macro)} duplicados")

# ===== LIMPIEZA FINAL DE COLUMNAS AUXILIARES =====
macro = macro.drop(
    columns=[
        "product_name_norm",
        "brand_norm",
        "subcategory_norm",
        "duplicado_evidente"
    ],
    errors="ignore"
)

# ===== ORDENAR COLUMNAS =====
ordered_cols = [
    "source", "product_name", "brand", "category", "subcategory",
    "price", "price_original", "discount_pct", "rating",
    "review_count", "popularity", "seller", "promo_tag",
    "product_url", "sku", "url_mal_formateada", "duplicado_url"
]
macro = macro[[c for c in ordered_cols if c in macro.columns]].copy()

# ===== GUARDAR =====
macro.to_csv(macro_file, index=False, encoding="utf-8-sig")

print(f"\n{'='*60}")
print(f"✅ DATASET DEFINITIVO CREADO")
print(f"{'='*60}")
print(f"📁 Guardado en: {macro_file}")
print(f"📊 Filas totales: {len(macro)}")
print(f"\n=== DISTRIBUCIÓN POR FUENTE ===")
print(macro['source'].value_counts())
print(f"\n=== RESUMEN DE CALIDAD ===")
print(f"Productos con rating: {macro['rating'].notna().sum()} ({macro['rating'].notna().sum()/len(macro)*100:.1f}%)")
print(f"Productos con review_count: {macro['review_count'].notna().sum()} ({macro['review_count'].notna().sum()/len(macro)*100:.1f}%)")
print(f"\n=== RATINGS POR FUENTE ===")
print(macro.groupby('source')['rating'].agg(['count', 'mean', 'min', 'max']))
print(f"\n=== Primeras 5 filas ===")
print(macro[['source', 'product_name', 'brand', 'subcategory', 'price', 'rating', 'review_count']].head())

📥 Cargando datasets...
  - PC Componentes: 231 productos
  - Kaggle Amazon: 29 productos

🔧 Estandarizando PC Componentes...
🔧 Estandarizando Kaggle Amazon...

🔗 Concatenando datasets...
🔗 Normalizando URLs...
🗑️ Eliminadas 29 filas sin nombre o precio
🗑️ Eliminados 0 duplicados

✅ DATASET DEFINITIVO CREADO
📁 Guardado en: C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Limpieza de datos y scrapping\Dataset Definitivo.csv
📊 Filas totales: 231

=== DISTRIBUCIÓN POR FUENTE ===
source
pccomponentes    231
Name: count, dtype: Int64

=== RESUMEN DE CALIDAD ===
Productos con rating: 231 (100.0%)
Productos con review_count: 231 (100.0%)

=== RATINGS POR FUENTE ===
               count      mean  min  max
source                                  
pccomponentes    231  4.548485  3.0  5.0

=== Primeras 5 filas ===
          source                                       product_name  brand  \
0  pccomponentes  PcCom Ready AMD Ryzen 7 5800X / 32GB / 1TB SSD...  PcCom   
1